# RSNA Knee — efficiency submission

**exp008**: DINOv2 ViT-S/14, T=16 slices, 320px, distilled (KL+BCE mix against the 3-member ensemble's soft predictions + `labels_v3_candidate`) — CLAUDE.md §8 efficiency-track model. Accepted 2026-09-10: OOF macro vs `labels_v2_final` 0.8708, only -0.0036 off the full ensemble (0.8744), well inside the 0.015 target.

**fp16, not bf16** — T4 (the pinned accelerator, T4 x2) is sm_75 with no native bf16 tensor cores; this notebook has always passed fp16 explicitly (`AUTOCAST_DTYPE`, cell-3), so no change needed here for the 2026-09-11 T4 adaptation. **No TTA** (rejected for the ensemble on OOF grounds; not re-tested for exp008, but there's no reason to expect a different verdict at a different resolution — not worth the 2x cost either way for the efficiency lane specifically).

**Batched decode**: DICOM decode (`decode_one_for_pool`/`decode_study_slots`, pydicom-based, CPU-bound) is parallelized across worker THREADS (not processes — see cell-2 for the 2026-09-10 finding that a process pool deadlocks after CUDA init and can't locate the worker function through this notebook's custom package loader; threads sidestep both and still parallelize the GIL-releasing C-extension work: DICOM decompression and numpy/scipy resampling).

**Decode/inference overlap** (2026-09-11, `infer_core.prefetch_decode`, cell-8): study N+1 now decodes in a background thread pool while study N runs on the GPU, instead of the old decode-a-whole-chunk-then-infer-the-whole-chunk lockstep (which left the GPU idle for an entire chunk's decode phase and the CPU idle for an entire chunk's inference phase).

**Decode fast path** (2026-09-10, `FAST_DECODE` in cell-3): profiling found decode is ~82% of this notebook's per-study cost at T=16 — a full-series decode (~30 slices) discards roughly half its slices to the post-decode T=16 subsample. `decode_one_for_pool_fast`/`decode_series_to_uint8_fast` reads DICOM headers only (no pixel decompression) to order the series, then decodes pixel data only for the T slices actually used. Validated against the accurate path on the full 1300-study synthetic load test before being turned on by default (EXPERIMENTS.md 2026-09-10) — the one approximation it makes is percentile normalization computed over T slices instead of the full series.

**Dual-GPU: NOT used here, and that's deliberate** (2026-09-11 T4 x2 audit) — the ensemble notebook shards its 3 members across cuda:0/cuda:1 because model inference there is a genuinely large fraction of per-study cost (~32% on H100). Here, decode is ~82% of per-study cost and model inference only ~18% — even an optimistic 2-way split of that small model slice saves only a couple of minutes out of a projected ~13-24 min total, while adding real complexity and Kaggle-memory risk. Single GPU + the decode/inference overlap above already captures nearly all the achievable gain; **pin whichever single T4 Kaggle offers this notebook** (or just leave the 2nd GPU idle if the session provides T4 x2 to both notebooks — harmless, just unused here).

**Fold count is chosen by the CLAUDE.md §4 efficiency-track FORMULA** (`Eff = AUC/(0.5−maxAUC) + s/32400`, minimized), not by "does it fit a time budget" (2026-09-10 correction — that conflated the CLAUDE.md §8 45min *target* with the wall-clock crash-prevention budget, which is now correctly the true 8h Kaggle cap in `BUDGET_SECONDS`). Cell-6 measures decode and model time SEPARATELY in-notebook on Kaggle's actual GPU, projects total runtime for each candidate fold count (1-5), looks up that count's OOF macro-AUC (gold-58 subset-averaging deltas vs. the accepted full-ensemble 0.8708 — see cell-3), and picks whichever fold count minimizes the formula. Because decode (fold-count-independent) dominates total runtime so heavily, the runtime term barely moves between fold counts.

**Runtime is a PROJECTION** from a calibration pass on the first studies — not yet backed by a real 1300-study run on the actual Kaggle accelerator. T4 projection (3.5x model-time multiplier applied to the H100-measured model cost, decode unchanged — CPU-bound, already shown CPU-count-insensitive): see `EXPERIMENTS.md` 2026-09-11 for the full table.

Expects attached Kaggle Datasets: `rsna-knee-code`, `rsna-knee-exp008-weights`, plus the competition data. Internet OFF.

In [ ]:
import importlib.util
import os, sys, time
from concurrent.futures import ThreadPoolExecutor, as_completed
from pathlib import Path

import pandas as pd
import torch

def _find_dir(root: str, name_contains: str, max_depth: int = 3) -> Path | None:
    # 2026-09-16 fix #2: the shallow iterdir() fix above (2026-09-16 fix #1, for the 84-min
    # rglob) was itself wrong on real Kaggle hardware -- CODE_DIR came back None. Kaggle's
    # actual mount layout nests attachments under /kaggle/input/datasets/hrdevelops/<dataset>
    # and /kaggle/input/competitions/<name>, not flat under /kaggle/input/<name>/ as assumed.
    # Checks the known Kaggle path shapes first (cheap, no directory listing), then falls back
    # to a depth-bounded BFS (max_depth=3, pruning known-huge DICOM series directories) so an
    # unrecognized mount layout still resolves without repeating the original full-tree-walk cost.
    root_path = Path(root)
    if not root_path.exists():
        return None
    candidates = [
        root_path / name_contains,
        root_path / "competitions" / name_contains,
        root_path / "datasets" / name_contains,
        root_path / "datasets" / "hrdevelops" / name_contains,
    ]
    for c in candidates:
        if c.exists() and c.is_dir():
            return c
    queue = [(root_path, 0)]
    while queue:
        curr, depth = queue.pop(0)
        if depth > 0 and name_contains in curr.name:
            return curr
        if depth < max_depth:
            try:
                for entry in curr.iterdir():
                    if entry.is_dir():
                        if name_contains in entry.name:
                            return entry
                        if entry.name not in ("test_series", "train_series", "series", "images"):
                            queue.append((entry, depth + 1))
            except (PermissionError, OSError):
                pass
    return None

CODE_DIR = _find_dir("/kaggle/input", "rsna-knee-code")
TEST_DIR = _find_dir("/kaggle/input", "rsna-knee-abnormality-detection")
WEIGHTS_DIR = _find_dir("/kaggle/input", "rsna-knee-exp008-weights")
OUT_PATH = Path("/kaggle/working/submission.csv")

print("CODE_DIR:", CODE_DIR)
print("TEST_DIR:", TEST_DIR)
print("WEIGHTS_DIR:", WEIGHTS_DIR)
assert CODE_DIR is not None, "Attach rsna-knee-code"
assert TEST_DIR is not None, "Attach the competition data"
assert WEIGHTS_DIR is not None, "Attach rsna-knee-exp008-weights"

init_files = list(CODE_DIR.rglob("__init__.py"))
assert init_files, f"No __init__.py found under {CODE_DIR}"
pkg_dir = init_files[0].parent
spec = importlib.util.spec_from_file_location("rsna_knee", pkg_dir / "__init__.py", submodule_search_locations=[str(pkg_dir)])
rsna_knee = importlib.util.module_from_spec(spec)
sys.modules["rsna_knee"] = rsna_knee
spec.loader.exec_module(rsna_knee)

print("torch", torch.__version__, "cuda available:", torch.cuda.is_available(),
      "device count:", torch.cuda.device_count() if torch.cuda.is_available() else 0)

In [ ]:
from rsna_knee.dataset import TARGET_COLUMNS, _patch_size_for_backbone
from rsna_knee.infer_core import (
    WallClockGuard, decode_one_for_pool, decode_one_for_pool_fast, nearest_patch_multiple,
    prefetch_decode, prior_row, run_study_inference,
)
from rsna_knee.model import SlotModel

# Batched decode uses ThreadPoolExecutor, not a process pool (2026-09-10 finding): a process pool
# hit two compounding problems here — forking after CUDA is initialized in the parent deadlocks
# the child, and even a spawn context can't find decode_one_for_pool because the custom
# sys.modules-injection package loader above isn't visible to a freshly-spawned interpreter.
# Threads sidestep both (shared process, no pickling) and still parallelize the expensive part of
# decode, since DICOM decompression (pylibjpeg/gdcm) and numpy/scipy resampling are C extensions
# that release the GIL. decode_one_for_pool itself still lives in the package (not this notebook)
# so it stays usable from either executor type without further changes.

In [ ]:
BACKBONE = "vit_small_patch14_dinov2.lvd142m"
SLICES_PER_SERIES = 16
IMAGE_SIZE = 320
OUT_SIZE = nearest_patch_multiple(IMAGE_SIZE, _patch_size_for_backbone(BACKBONE))  # 322, patch14 rounding

AUTOCAST_DTYPE = torch.float16  # fp16, not bf16 — see markdown
DECODE_CHUNK_SIZE = 24          # studies decoded per batch
# os.cpu_count() can report a shared host's FULL core count rather than this container's actual
# allocation in some environments (found 2026-09-10 on HPC SLURM: requested 16 cores, cpu_count()
# reported 192) — capped defensively rather than trusted blindly. Kaggle containers aren't shared
# SLURM nodes, so os.cpu_count() here should already be correct (commonly 4 CPUs) — the cap just
# costs nothing to keep. Threads (not processes, see cell-2) make an overshoot far cheaper than it
# would be with a process pool regardless.
DECODE_WORKERS = max(1, min((os.cpu_count() or 4) - 1, 32))

# Header-only ordering + T-slice-only pixel decode instead of full-series decode (2026-09-10):
# profiling found decode is ~82% of this notebook's per-study cost at T=16 (model inference only
# ~18%), almost entirely wasted since a typical ~30-slice series gets fully decoded only to have
# roughly half its slices thrown away by the post-decode linspace subsample. Validated against the
# accurate path on the full 1300-study synthetic load test (EXPERIMENTS.md 2026-09-10) before
# being turned on by default here — see preprocess.decode_series_to_uint8_fast for exactly what
# approximation this makes (percentile normalization over T slices instead of the full series).
FAST_DECODE = True

# 8h true Kaggle wall-clock safety cap (2026-09-10 correction) — NOT the CLAUDE.md §8 45min
# *target*, which was wrongly used as this budget before: that number is a performance goal for
# the efficiency-track SCORE (cell-7 now picks the fold count that's actually formula-optimal for
# that score), not the crash-prevention budget, which should be the same true cap the ensemble
# notebook uses.
BUDGET_SECONDS = 8 * 3600
CALIBRATION_STUDIES = 10

# OOF macro-AUC by fold count — measured 2026-09-10 on gold-58 (the only set where every fold's
# model predicts every study): averaging over ALL subsets of each size (not just the best) gives
# mean deltas vs. the accepted full 5-fold OOF (0.8708, vs labels_v2_final) of -0.0060 (1 fold),
# -0.0030 (2), -0.0014 (3), -0.0004 (4), 0 (5 — the full ensemble). Feeds the efficiency formula
# below as the AUC term for each candidate fold count.
OOF_MACRO_AUC_5FOLD = 0.8708
FOLD_COUNT_AUC_DELTA = {1: -0.0060, 2: -0.0030, 3: -0.0014, 4: -0.0004, 5: 0.0}
OOF_AUC_BY_FOLDS = {k: OOF_MACRO_AUC_5FOLD + d for k, d in FOLD_COUNT_AUC_DELTA.items()}

# CLAUDE.md §4: Efficiency = AUC / (0.5 - maxAUC) + RuntimeSeconds / 32400, minimized. maxAUC is
# the competition's best private-LB AUC, unknown until the leaderboard closes — 0.92 is our
# working assumption (2026-09-10 instruction) until it can be measured for real.
ASSUMED_MAX_AUC = 0.92
EFF_DENOM = 0.5 - ASSUMED_MAX_AUC  # -0.42, negative -> higher AUC LOWERS (improves) the score


def efficiency_score(auc: float, runtime_seconds: float) -> float:
    return auc / EFF_DENOM + runtime_seconds / 32400


DEVICE = "cpu"
if torch.cuda.is_available():
    major, minor = torch.cuda.get_device_capability(0)
    cap = major * 10 + minor
    supported_caps = [int(a[3:]) for a in torch.cuda.get_arch_list() if a.startswith("sm_")]
    if supported_caps and cap >= min(supported_caps):
        DEVICE = "cuda"
    else:
        print(f"  [WARN] GPU {torch.cuda.get_device_name(0)} (sm_{cap}) unsupported by this torch "
              f"build (min sm_{min(supported_caps) if supported_caps else '?'}) — CPU fallback.")
print("device:", DEVICE, "out_size:", OUT_SIZE, "decode_workers:", DECODE_WORKERS, "fast_decode:", FAST_DECODE)

In [ ]:
def load_fold_models(weights_dir: Path, backbone: str, device: str, n_folds: int) -> list:
    # Flat foldN_best.pt layout (2026-09-11) — see submit-ensemble.ipynb cell-3 for why: per-fold
    # SUBDIRECTORIES need --dir-mode zip to upload at all, and Kaggle's zip-based subdirectory
    # extraction has previously been found (2026-09-08 arc, EXPERIMENTS.md) to not reliably
    # preserve folder nesting — harmless for the code package's unique filenames, but every fold's
    # checkpoint is identically named "best.pt", so a flattening extraction could collide 5 files
    # into 1. Falls back to the foldN/best.pt subdirectory layout for robustness.
    ckpt_paths = sorted(weights_dir.glob("fold*_best.pt"))[:n_folds]
    if not ckpt_paths:
        ckpt_paths = [d / "best.pt" for d in sorted(weights_dir.glob("fold*"))[:n_folds]
                      if d.is_dir() and (d / "best.pt").exists()]
    models = []
    for ckpt_path in ckpt_paths:
        if not ckpt_path.exists():
            print(f"  [WARN] no checkpoint at {ckpt_path}")
            continue
        ckpt = torch.load(ckpt_path, map_location=device, weights_only=False)
        model = SlotModel(backbone, num_targets=len(TARGET_COLUMNS), pretrained=False).to(device)
        model.load_state_dict(ckpt["ema"])
        model.eval()
        models.append(model)
    return models

ALL_MODELS = load_fold_models(WEIGHTS_DIR, BACKBONE, DEVICE, n_folds=5)
print(f"Loaded {len(ALL_MODELS)}/5 folds.")
if not ALL_MODELS:
    raise RuntimeError("No fold models loaded — cannot run inference.")

In [ ]:
test_df = pd.read_csv(TEST_DIR / "test.csv")
series_df = pd.read_csv(TEST_DIR / "test_series.csv").rename(
    columns={"StudyInstanceUID": "study_uid", "SeriesInstanceUID": "series_uid"}
)
series_root = TEST_DIR / "test_series"
print(f"{len(test_df)} test studies, {len(series_df)} series.")

In [ ]:
_decode_worker_fn = decode_one_for_pool_fast if FAST_DECODE else decode_one_for_pool


def decode_chunk(study_uids: list[str]) -> dict[str, tuple[dict, list, str | None]]:
    """Decodes a chunk of studies in parallel worker THREADS (see cell-2 for why not processes).
    Returns {study_uid: (slots, tag_sides, error_or_None)} — one chunk's volumes are held in
    memory at a time (T=16/320px is small per study, but 1300 studies x 6 slots at once would not
    be). Uses decode_one_for_pool_fast when FAST_DECODE is set (cell-3)."""
    tasks = [
        (uid, series_df[series_df["study_uid"] == uid].to_dict("records"), str(series_root), SLICES_PER_SERIES, OUT_SIZE)
        for uid in study_uids
    ]
    results = {}
    with ThreadPoolExecutor(max_workers=DECODE_WORKERS) as pool:
        futures = {pool.submit(_decode_worker_fn, t): t[0] for t in tasks}
        for fut in as_completed(futures):
            uid, slots, tag_sides, err = fut.result()
            results[uid] = (slots, tag_sides, err)
    return results

In [ ]:
# Fold count is chosen to MINIMIZE the CLAUDE.md §4 efficiency-track formula, not just to fit a
# wall-clock budget (2026-09-10 correction) — the wall-clock guard below is now purely a
# crash-prevention safety net (BUDGET_SECONDS = true 8h Kaggle cap), separate from this decision.
guard = WallClockGuard(BUDGET_SECONDS)

# --- Calibration: decode+infer the first studies at full (5-fold) fidelity, timing decode and
# model inference SEPARATELY so the fold-count projection below is a real in-notebook measurement
# on Kaggle's actual GPU (not an HPC assumption) — decode cost doesn't scale with fold count,
# model cost does, so conflating them into one combined per-study number (as an earlier version of
# this cell did) would misproject every rung except 5-fold.
n_calib = min(CALIBRATION_STUDIES, len(test_df))
calib_uids = list(test_df["StudyInstanceUID"].iloc[:n_calib])

decode_t0 = time.time()
calib_decoded = decode_chunk(calib_uids)
decode_calib_elapsed = time.time() - decode_t0

model_t0 = time.time()
calib_rows = []
for uid in calib_uids:
    slots, tag_sides, err = calib_decoded[uid]
    if slots and err is None:
        probs = run_study_inference(slots, tag_sides, ALL_MODELS, DEVICE, SLICES_PER_SERIES, OUT_SIZE, AUTOCAST_DTYPE)
        calib_rows.append({"StudyInstanceUID": uid, **{c: float(p) for c, p in zip(TARGET_COLUMNS, probs)}})
    else:
        print(f"  [WARN] calibration decode failed for {uid}: {err}")
model_calib_elapsed = time.time() - model_t0

decode_per_study = decode_calib_elapsed / max(n_calib, 1)
model_per_study_5fold = model_calib_elapsed / max(n_calib, 1)
gpu_name = torch.cuda.get_device_name(0) if DEVICE == "cuda" else "CPU"
print(f"Calibration: {n_calib} studies on {gpu_name} — decode {decode_calib_elapsed:.1f}s "
      f"({decode_per_study:.3f}s/study), model-5fold {model_calib_elapsed:.1f}s ({model_per_study_5fold:.3f}s/study)")

print(f"\nFold-count selection — efficiency formula (maxAUC assumed {ASSUMED_MAX_AUC}):")
eff_by_folds = {}
for k in sorted(OOF_AUC_BY_FOLDS):
    projected_per_study = decode_per_study + model_per_study_5fold * (k / 5)
    projected_total = projected_per_study * len(test_df)
    auc = OOF_AUC_BY_FOLDS[k]
    score = efficiency_score(auc, projected_total)
    fits = projected_total <= guard.budget * 0.85
    eff_by_folds[k] = (score, projected_total, fits)
    print(f"  {k} fold(s): OOF AUC {auc:.4f}, projected {projected_total / 60:.1f} min "
          f"({'fits' if fits else 'EXCEEDS 8h BUDGET — excluded'} the 8h safety cap), "
          f"efficiency score {score:.4f}")

# Pick the formula-optimal fold count among rungs that safely fit the true wall-clock budget (in
# practice every rung fits easily — even 5-fold projects to well under an hour on any Kaggle GPU
# — so this is a defensive filter, not the actual selection driver).
feasible = {k: v for k, v in eff_by_folds.items() if v[2]}
chosen_n_folds = min(feasible, key=lambda k: feasible[k][0]) if feasible else min(eff_by_folds, key=lambda k: eff_by_folds[k][1])
print(f"\nChosen fold count: {chosen_n_folds}/5 (efficiency score {eff_by_folds[chosen_n_folds][0]:.4f})")

ACTIVE_MODELS = ALL_MODELS[:chosen_n_folds]

In [ ]:
rows = list(calib_rows) if chosen_n_folds == 5 else []  # reuse calibration only if it matches the chosen rung
done_uids = {r["StudyInstanceUID"] for r in rows}
remaining_uids = [u for u in test_df["StudyInstanceUID"] if u not in done_uids]


def _task_for(uid: str) -> tuple:
    return (uid, series_df[series_df["study_uid"] == uid].to_dict("records"), str(series_root),
            SLICES_PER_SERIES, OUT_SIZE)


remaining_tasks = [_task_for(u) for u in remaining_uids]

# Decode/inference overlap (2026-09-11 T4 adaptation): study N+1 decodes in a background thread
# pool (prefetch_decode) while study N runs on the GPU, instead of the old decode-a-whole-chunk-
# then-infer-the-whole-chunk lockstep, which left the GPU idle for an entire chunk's decode phase
# and the CPU idle for an entire chunk's inference phase. Dual-GPU model sharding was profiled and
# rejected for THIS notebook (see cell-0 markdown) — decode so dominates per-study cost here that
# splitting the small model-inference slice across 2 GPUs barely moves the total; single GPU + this
# overlap already captures nearly all the achievable gain.
n_prior_fallback = n_dual_flip = 0
processed = len(rows)
for uid, slots, tag_sides, err in prefetch_decode(remaining_tasks, _decode_worker_fn, DECODE_WORKERS):
    if guard.over(margin=0.05):
        break
    row = {"StudyInstanceUID": uid}
    try:
        if err is not None or not slots:
            raise RuntimeError(err or "no series decoded successfully for this study")
        if all(s not in ("R", "L") for s in tag_sides):
            n_dual_flip += 1
        probs = run_study_inference(slots, tag_sides, ACTIVE_MODELS, DEVICE, SLICES_PER_SERIES, OUT_SIZE, AUTOCAST_DTYPE)
        row.update({c: float(p) for c, p in zip(TARGET_COLUMNS, probs)})
    except Exception as e:
        print(f"  [WARN] {uid}: {type(e).__name__}: {e} — using label priors")
        n_prior_fallback += 1
        row.update({c: float(p) for c, p in zip(TARGET_COLUMNS, prior_row())})
    rows.append(row)
    processed += 1
    if processed % DECODE_CHUNK_SIZE == 0 or processed == len(test_df):
        print(f"  {processed}/{len(test_df)} studies, {guard.elapsed():.0f}s elapsed ({guard.fraction_used():.1%} of budget)")

if processed < len(test_df):
    done_now = {r["StudyInstanceUID"] for r in rows}
    skipped = [u for u in remaining_uids if u not in done_now]
    print(f"  [BUDGET] {guard.elapsed():.0f}s used, {processed}/{len(test_df)} done — priors for the rest.")
    for uid in skipped:
        rows.append({"StudyInstanceUID": uid, **{c: float(p) for c, p in zip(TARGET_COLUMNS, prior_row())}})
        n_prior_fallback += 1

print(f"Done: {len(rows)} studies, fold_count={chosen_n_folds}, {n_prior_fallback} full-prior fallbacks, "
      f"{n_dual_flip} dual-flip, {guard.elapsed():.0f}s total.")

In [ ]:
sub_df = pd.DataFrame(rows)[["StudyInstanceUID"] + TARGET_COLUMNS]

assert len(sub_df) == len(test_df), f"row count mismatch: {len(sub_df)} vs {len(test_df)}"
assert set(sub_df["StudyInstanceUID"]) == set(test_df["StudyInstanceUID"]), "study UID mismatch"
assert not sub_df[TARGET_COLUMNS].isna().any().any(), "NaNs in submission"
assert (sub_df[TARGET_COLUMNS] >= 0).all().all() and (sub_df[TARGET_COLUMNS] <= 1).all().all(), "probabilities out of [0,1]"
assert list(sub_df.columns) == ["StudyInstanceUID"] + TARGET_COLUMNS, f"header mismatch: {list(sub_df.columns)}"

sub_df.to_csv(OUT_PATH, index=False)
print(f"Wrote {OUT_PATH}: {len(sub_df)} rows, all checks passed. Folds used: {chosen_n_folds}/5.")
sub_df.head()